In [3]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from transformers import AutoImageProcessor, AutoModel
import torch.nn as nn
import os
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from tqdm.notebook import tqdm
from icecream import ic
from transformers import get_cosine_schedule_with_warmup
from PIL import Image
import random
import numpy as np
import timm
from vit_tiny_main import vit_tiny_classifier

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)



In [4]:


class OxfordPetDataset(Dataset):
    def __init__(self, root_dir, split="trainval", transform=None):
        """
        root_dir: path to 'oxford-iiit-pet'
        split: 'trainval' or 'test'
        transform: torchvision transforms
        """
        self.root_dir = root_dir
        self.images_dir = os.path.join(root_dir, "images")
        self.annotations_file = os.path.join(root_dir, "annotations", f"{split}.txt")
        # self.transform = transform
        if transform is None:
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transform

        self.samples = []
        self._load_annotations()
        ic(len(self.samples))

    def _load_annotations(self):
        with open(self.annotations_file, "r") as f:
            for line in tqdm(f):
                parts = line.strip().split()
                image_name = parts[0] + ".jpg"
                label = int(parts[1]) - 1  # convert to 0-based index
                
                self.samples.append((image_name, label))
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_name, label = self.samples[idx]

        image_path = os.path.join(self.images_dir, image_name)
        image = Image.open(image_path).convert("RGB")
        image = image.resize((224, 224))

        if self.transform:
            image = self.transform(image)

        return image, label


train_dataset = OxfordPetDataset(root_dir="/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/data/oxford-iiit-pet", split="trainval")
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,      # How many samples per batch
    shuffle=True)

test_dataset = OxfordPetDataset(root_dir="/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/data/oxford-iiit-pet", split="test")
test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,      # How many samples per batch
    shuffle=False)


0it [00:00, ?it/s]

ic| len(self.samples): 3680


0it [00:00, ?it/s]

ic| len(self.samples): 3669


In [5]:
for images, labels in train_dataloader:
    print("Batch of images shape:", images.shape)
    print("Batch of labels shape:", labels.shape)
    print(labels)
    break

Batch of images shape: torch.Size([32, 3, 224, 224])
Batch of labels shape: torch.Size([32])
tensor([20, 14, 31,  6, 30, 33, 20, 25, 31,  2, 28,  7,  9, 24, 14, 12, 35, 20,
        22, 24, 32,  3, 10,  9,  1, 13, 11, 33,  9, 12, 11,  0])


In [6]:
class ViT_Wrapper(nn.Module):
    def __init__(self, model, n_classes = 37):
        super().__init__()
        # this will give a feature vector of 1000 dim, put a linear layer to transfer from 1000 -> n_classes
        self.model = model
        self.mlp = nn.Sequential(
            nn.Linear(1000, 512),
            nn.GELU(),
            nn.Linear(512, n_classes)
        )

        self.freeze()
    
    def freeze(self):
        for p in self.model.parameters():
            p.requires_grad = False
        
    def forward(self, x):
        feat = self.model(x)
        logits = self.mlp(feat)
        return logits
vit_tiny = vit_tiny_classifier(n_classes=1000)
checkpoint = torch.load("/media/system/ZERBUIS_EXT_STOR/dynamic_slam/src/vit_tiny_checkpoints/model_epoch_76.pth")
vit_tiny.load_state_dict(checkpoint['model_state_dict'])
model = ViT_Wrapper(vit_tiny)

In [7]:
BATCH_SIZE = 32
LR = 5e-5
EPOCHS = 10
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05

device = "cuda"

model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_training_steps = EPOCHS * len(train_dataloader)
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

criterion = nn.CrossEntropyLoss()

for epoch in tqdm(range(EPOCHS)):

    # ---------------- TRAIN ----------------
    model.train()
    total_loss = 0
    correct, total = 0, 0

    for images, labels in train_dataloader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)           # (B, num_classes)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    avg_loss = total_loss / len(train_dataloader)

    # ---------------- VALID ----------------
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in test_dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {avg_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Acc: {val_acc:.2f}%"
    )

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch [1/10] Loss: 3.8760 Train Acc: 6.14% Val Acc: 18.15%
Epoch [2/10] Loss: 3.1639 Train Acc: 18.72% Val Acc: 24.39%
Epoch [3/10] Loss: 2.7614 Train Acc: 33.53% Val Acc: 45.03%
Epoch [4/10] Loss: 2.4296 Train Acc: 48.86% Val Acc: 58.68%
Epoch [5/10] Loss: 2.1777 Train Acc: 57.26% Val Acc: 58.63%
Epoch [6/10] Loss: 1.9784 Train Acc: 64.54% Val Acc: 70.54%
Epoch [7/10] Loss: 1.8491 Train Acc: 70.19% Val Acc: 69.53%
Epoch [8/10] Loss: 1.7799 Train Acc: 71.85% Val Acc: 73.54%
Epoch [9/10] Loss: 1.7345 Train Acc: 75.24% Val Acc: 76.59%
Epoch [10/10] Loss: 1.7183 Train Acc: 76.06% Val Acc: 76.61%


: 